In [1]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import math

In [2]:
FILE = '../data/cleaned_output.csv'

"""
raw df has continous time series data for each user, no gap between days.
1 sugg.select.utime is Nan: invalid
Number of unique users: 37
"""
df = pd.read_csv(FILE)
users = df['uid'].unique()
print(df.columns)

df['datetime'] =  pd.to_datetime(df['datetime'])
df['is_weekend'] = df['datetime'].dt.dayofweek >= 5

df['is_weekend'].value_counts()

Index(['datetime', 'uid', 'decision_idx', 'date', 'day_slot', 'is_randomized',
       'avail', 'send', 'returned_message', 'response', 'activity', 'location',
       'weather', 'temperature', 'jbsteps10', 'jbsteps30', 'jbsteps40',
       'jbsteps60', 'jbsteps90', 'jbsteps120', 'jbsteps30pre', 'jbsteps40pre',
       'jbsteps60pre'],
      dtype='str')


is_weekend
False    5178
True     2057
Name: count, dtype: int64

In [13]:
print(df.iloc[df['is_weekend']==0].groupby(['uid', 'day_slot'])['location'].value_counts().to_string())

uid  day_slot  location                                        
1    1         Home                                                23
               Food & Dining                                        1
     2         Work                                                17
               Home                                                 5
               Food & Dining                                        1
               Auto & Transport - Outdoor Low-Activity              1
               University/School                                    1
               Healthcare & Personal Care - Indoor Low-Activity     1
     3         Work                                                15
               Home                                                 6
               University/School                                    2
               Clothing & Fashion Store                             1
               Home & Furniture Store                               1
               Healthcare 

In [15]:
print(df.iloc[df['is_weekend']==1].groupby(['uid', 'day_slot'])['location'].value_counts().to_string())

uid  day_slot  location                                        
1    1         Home                                                 8
               Food & Dining                                        2
     2         Home                                                 7
               Auto & Transport - Outdoor Low-Activity              1
               Home & Furniture Store                               1
               Food & Dining                                        1
     3         Home                                                 6
               Clothing & Fashion Store                             1
               Just Store                                           1
               Food & Dining                                        1
               Other Specialty Store                                1
     4         Home                                                 7
               Just Store                                           1
               Food & Dini

In [27]:
print(df.iloc[df['is_weekend']==0].groupby(['uid', 'day_slot'])['jbsteps30pre'].value_counts().to_string())

uid  day_slot
1    1           24
     2           26
     3           26
     4           26
     5           25
2    1           25
     2           21
     3           22
     4           22
     5           24
3    1           29
     2           28
     3           26
     4           28
     5           27
4    1           29
     2           30
     3           30
     4           28
     5           28
5    1           31
     2           30
     3           32
     4           32
     5           31
6    1           25
     2           25
     3           23
     4           24
     5           25
7    1           27
     2           32
     3           31
     4           28
     5           29
8    1           30
     2           31
     3           33
     4           31
     5           31
9    1           29
     2           27
     3           28
     4           29
     5           30
10   1           31
     2           31
     3           30
     4           31
     5

In [26]:
print(df.iloc[df['is_weekend']==1].groupby(['uid', 'day_slot'])['jbsteps30pre'].value_counts().to_string())

uid  day_slot  jbsteps30pre
1    1         0               10
     2         0                3
               55               1
               1484             1
               595              1
               858              1
               1051             1
               29               1
               311              1
     3         0                2
               1270             1
               863              1
               454              1
               431              1
               1189             1
               369              1
               556              1
               343              1
     4         0                2
               245              1
               1011             1
               490              1
               759              1
               750              1
               273              1
               127              1
               264              1
     5         0                2
               353  

In [39]:
df['jbsteps30diff'] = round((df['jbsteps30'] - df['jbsteps30pre']) / df['jbsteps30pre'] * 100, 3)
print(df.iloc[df['is_weekend']==1 & (df['send']!=0)].groupby(['uid', 'day_slot'])['jbsteps30diff'].value_counts().to_string())

uid  day_slot  jbsteps30diff
1    1          inf              6
     2         -100.000          2
                49.091           1
               -22.709           1
               -44.346           1
               -69.244           1
                59.207           1
                40.278           1
               -51.895           1
               -11.893           1
                180.583          1
                3489.474         1
                401.370          1
     3         -100.000          3
               -53.937           1
                37.659           1
               -41.127           1
               -87.534           1
                inf              1
                78.523           1
                47.662           1
               -83.021           1
     4          30.651           1
               -79.822           1
                556.790          1
               -20.379           1
                8.571            1
                12.717    

In [36]:
print(df.iloc[df['is_weekend']==0 & (df['send']!=0)].groupby(['uid', 'day_slot'])['jbsteps30diff'].value_counts().to_string())

uid  day_slot  jbsteps30diff
1    1          0                9
                254              1
                463              1
                62               1
                584              1
                456              1
                551              1
                160              1
                291              1
                315              1
                364              1
                807              1
                838              1
                946              1
                298              1
                21               1
     2          0                3
                1311             1
               -275              1
               -191              1
               -24               1
                218              1
                1899             1
               -276              1
               -200              1
               -19               1
                87               1
               -356       

In [23]:
user_df = pd.read_csv('../data/raw_data/users.csv')
print(user_df.groupby(['occupation'])[['user.index','age', 'gender']].value_counts().to_string())


occupation                                              user.index  age  gender
IT manager                                              16          55   male      1
administrative assistant                                26          53   female    1
clerk, deputy                                           14          54   female    1
customer service (Comcast)                              34          64   female    1
engineer                                                37          47   female    1
                                                        6           20   male      1
engineering intern, lifegaurd                           4           21   male      1
environmental health specialist                         25          46   female    1
grad student                                            12          25   male      1
graduate student                                        31          26   male      1
healthcare researcher                                   21          28